In [2]:
import requests
import pandas as pd

In [3]:
# dictionary of all teams:
teams = requests.get('https://statsapi.mlb.com/api/v1/teams', params={'division' : 'Major League Baseball'}).json()

In [4]:
# hardcoded api keys:
intl_league_id = 117

In [5]:
def get_league_team_ids(team_dict, league_id):
    print(f'Fetching list of teams in league {league_id}:')
    print('----------------------------------------------')
    team_id_list = []
    for team in team_dict['teams']:
        if 'id' in team['league'] and team['league']['id'] == league_id:
            print(team['name'])
            team_id_list.append(team['id'])
    return team_id_list

def get_team_roster_ids(team_id):
    roster = requests.get(f'https://statsapi.mlb.com/api/v1/teams/{team_id}/roster').json()
    player_id_list = []
    #print(roster['roster'])
    for player in roster['roster']:
        print(f'{player['person']['id']}  {player['person']['fullName']}')
        player_id_list.append(player['person']['id'])
    return player_id_list

def get_minor_logs(player_id, season):
    return requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats?stats=gameLog&gameType=R&leagueListId=milb_all&group=hitting&hydrate=team(league)&language=en&season={season}",).json()

def get_major_logs(player_id, season):
    return requests.get(f'https://statsapi.mlb.com/api/v1/people/{player_id}/stats?season={season}&group=hitting&stats=gameLog',).json()

def tabulate(log_file):
    pass
    # Hardcoded dictionary of positions?
    # position_list_raw_tallies = {'P': 0, etc.}
    # position_list_percent = {'P': None, etc.} # Cuz maybe I want an undefined flag for levels that they didn't play at?
    # Do I define these for every level? Or do I wait and call them into existence when they're needed? that's probably better
    # I need to figure out how best to stack this data into a tensor. That's the key design decision here.
    # A, AA, AAA, MLB, etc = []
    #for game in log_file[]: # This is a placeholder
        # Task 1: identify level (because the minor league logs are mixed I believe)
        # This code may have to handle MLB logs different from milb because they are differently structured
        # When a level comes up, add a tally to that count, then inspect the position, and add tallies as appropriate
    # Then, outside of loop, run a function against the raw position tallies to convert them to percentages for analysis
    # Thinking further, I wanted to exclude PH (and potentially pinch runner and other such things if they're recorded) but doing so may
    # cause a divide by zero in the edgecase where that's the only appearance at that level. And I DO want to record the appearance!
    # Ultimately, this:
    # return [data package that can slide into the tensor's seasonal columns]
    # The more I think about this the more I think I want to precollate the MLB and milb logs before they enter this function, would be so much cleaner

In [6]:
intl_league_team_ids = get_league_team_ids(teams, intl_league_id)
print(intl_league_team_ids)

Fetching list of teams in league 117:
----------------------------------------------
Durham Bulls
Memphis Redbirds
Louisville Bats
Buffalo Bisons
Gwinnett Stripers
Columbus Clippers
Iowa Cubs
Indianapolis Indians
Charlotte Knights
Toledo Mud Hens
Scranton/Wilkes-Barre RailRiders
Worcester Red Sox
Rochester Red Wings
Omaha Storm Chasers
Syracuse Mets
Nashville Sounds
Jacksonville Jumbo Shrimp
Norfolk Tides
Lehigh Valley IronPigs
St. Paul Saints
[234, 235, 416, 422, 431, 445, 451, 484, 494, 512, 531, 533, 534, 541, 552, 556, 564, 568, 1410, 1960]


In [7]:
intl_league_rosters = []
for team in intl_league_team_ids:
    team_roster = get_team_roster_ids(team)
    intl_league_rosters.append(team_roster)
intl_league_rosters

693739  Alexander Alberto
596103  Austin Slater
666165  Blake Sabol
805207  Caden Bodine
657611  Cam Hill
805363  Chris Clark
682126  Evan Reifert
683344  Homer Bush Jr.
669261  Jack Suwinski
666931  Jared Oliva
680701  John Rooney
687866  Justin Yeager
676831  Kodi Whitley
669722  Logan Davidson
694312  Logan Workman
687189  Marcus Johnson
686554  Oliver Dunn
801643  Owen Wild
694420  Tatem Levins
691003  Tre' Morgan
800060  Xavier Isaac
669697  Andrew Schultz
647378  Andy Yerzy
676153  Austin Love
676632  Bligh Madris
822410  Bobby Olsen
825576  Brandt Thompson
690914  Brody Moore
690971  Chase Davis
807742  Colton Ledbetter
703766  Dakota Harris
815104  Graysen Tarlow
676596  Josh Fleming
701388  Jurrangelo Cijntje
695766  Mason Molina
702331  Noah Mendlinger
677337  Pete Hansen
680636  Ramon Mendoza
694885  Ryan Murphy
811139  Sandro Santana
813766  Skylar Hales
813852  Trey Paige
804241  Zach Levenson
664948  Anthony Misiewicz
686844  Ben Wereski
605335  Brandon Leibrandt
694650  

[[693739,
  596103,
  666165,
  805207,
  657611,
  805363,
  682126,
  683344,
  669261,
  666931,
  680701,
  687866,
  676831,
  669722,
  694312,
  687189,
  686554,
  801643,
  694420,
  691003,
  800060],
 [669697,
  647378,
  676153,
  676632,
  822410,
  825576,
  690914,
  690971,
  807742,
  703766,
  815104,
  676596,
  701388,
  695766,
  702331,
  677337,
  680636,
  694885,
  811139,
  813766,
  813852,
  804241],
 [664948,
  686844,
  605335,
  694650,
  670241,
  802143,
  666150,
  694689,
  688005,
  641658,
  688609,
  671155,
  805723,
  695481,
  691641,
  592288,
  674265,
  656308,
  681114,
  606115,
  680397],
 [685121,
  667690,
  806413,
  663662,
  687288,
  804948,
  814804,
  694495,
  803385,
  688460,
  687069,
  695776,
  670174,
  681751,
  800959,
  693483,
  805268,
  702302,
  689041,
  682620,
  814217,
  700292],
 [700845,
  666768,
  822412,
  695462,
  678142,
  642020,
  680577,
  678173,
  805175,
  802357,
  622694,
  803833,
  803349,
  6658

In [9]:
len(intl_league_rosters)

20

In [10]:
intl_league_rosters[0]

[693739,
 596103,
 666165,
 805207,
 657611,
 805363,
 682126,
 683344,
 669261,
 666931,
 680701,
 687866,
 676831,
 669722,
 694312,
 687189,
 686554,
 801643,
 694420,
 691003,
 800060]